# 04 · The monthly download, and checking you got everything

**What this is for:** the job you run once a month, after a new forecast arrives. It downloads all nine bulk files (three levels × three outcomes) into a `data/` folder where Jupyter runs it (beside this notebook, when Jupyter is started in `notebooks/`), checks that each carries real values in every month, and ends with one verdict. Part B, to read once, explains what the files hold.

**Why the checks look at values.** A file can have the right size, the right rows and the right columns, and still be empty where it matters: a file whose last months carry nothing still downloads, still opens, and still reports the right months. So every check here reads the values.

**You need an API key** (set in `.env`, see `notebooks/README.md`), about 500 MB of disk, and about 5 GB of free memory (the checks read the grid files a few columns at a time; measured peak 5.0 GB).

# Part A · The monthly run

## 1 · Setup

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
import requests
from dotenv import load_dotenv

from views_crafdapi.client import CrafdApiClient
from views_crafdapi.data_checks import (
    PLACE_ID, DataCheckError, check_bulk_file_complete, check_files_agree, check_forecast_summaries,
)

load_dotenv()  # reads CRAFD_KEY (your key) and, optionally, VIEWS_API_URL

BASE_URL = os.environ.get("VIEWS_API_URL", "https://crafdapi.viewsforecasting.org")
API_KEY = os.environ.get("CRAFD_KEY", "").strip().strip('"')
# Optional, for checking a release: the number of draws every forecast row should come from.
# Unset, it is only shown.
try:
    EXPECTED_DRAWS = int(os.environ["CRAFD_EXPECTED_DRAWS"]) if os.environ.get("CRAFD_EXPECTED_DRAWS") else None
except ValueError:
    raise SystemExit("CRAFD_EXPECTED_DRAWS must be a whole number, such as the number of draws you were told.") from None
HEADERS = {"X-API-Key": API_KEY}
LEVELS = ("grid", "admin1", "country")
OUTCOMES = ("sb", "ns", "os")
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)

client = CrafdApiClient(BASE_URL, API_KEY, timeout=300)
print("Base URL :", BASE_URL)
print("API key  :", "set" if API_KEY else "MISSING: set it in .env")
print("Files go :", DATA_DIR)


def download_to(level, outcome):
    """Save one bulk file in data/ and return its path and the month the forecast was made in.
    The file is written beside its final name and renamed only when complete, so an interrupted
    download never leaves a file that looks whole. A busy server can answer 502 or 504, and a slow
    connection can time out or drop, and the service answers 503 for a minute or so while it
    loads a new forecast. Two retries, a minute apart, cover those."""
    final = DATA_DIR / f"{level}_{outcome}.parquet"
    partial = final.with_suffix(".part")
    for attempt in (1, 2, 3):
        try:
            resp = requests.get(f"{BASE_URL}/bulk/{level}/{outcome}", headers=HEADERS, timeout=600, stream=True)
            if resp.status_code in (502, 503, 504) and attempt < 3:
                resp.close()
                print(f"  the server answered {resp.status_code}; trying again in 60 seconds...")
                time.sleep(60)
                continue
            if not resp.ok:
                raise SystemExit(f"/bulk/{level}/{outcome} answered {resp.status_code}: {resp.text[:300]}")
            with resp, open(partial, "wb") as out:
                for chunk in resp.iter_content(chunk_size=1 << 20):
                    out.write(chunk)
            break
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError,
                requests.exceptions.ChunkedEncodingError):
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise
            print("  the download timed out or dropped; trying again in 60 seconds...")
            time.sleep(60)
    made_in = resp.headers.get("X-Forecast-Made-In")
    if not made_in:
        partial.unlink(missing_ok=True)
        raise SystemExit(f"/bulk/{level}/{outcome} did not say which month the forecast was made in.")
    partial.replace(final)
    return final, made_in

In [ ]:
if not API_KEY or API_KEY == "your-api-key-here":
    raise SystemExit("No usable API key. Set CRAFD_KEY in .env (see notebooks/README.md), restart "
                     "the kernel and run again.")
try:
    health = client.health()
except Exception as e:   # an error answer, a timeout or a dropped connection
    health = {}
    print("Health check unavailable, continuing:", e)
freshness = health.get("forecast_freshness") or {}
print("Service status :", health.get("status", "unknown"))
print("Run served     :", health.get("served_run") or "not reported")
print("Forecast age   :", freshness.get("age_days"), "days | the service's limit:", freshness.get("sla_days"), "days")
print("(07_is_the_data_current.ipynb checks whether this is the forecast you should have.)")

## 2 · Download the nine files

One file per level and outcome. The three grid files are about 130 MB each; the others are a few MB. If a download is interrupted, re-run this cell: a finished file is replaced only by a complete new one.

In [ ]:
paths, made_in = {}, {}
for level in LEVELS:
    for outcome in OUTCOMES:
        started = time.time()
        paths[level, outcome], made_in[level, outcome] = download_to(level, outcome)
        size = paths[level, outcome].stat().st_size / 1e6
        print(f"{level:8} {outcome}: {size:7.1f} MB in {time.time() - started:4.0f} s, forecast made in {made_in[level, outcome]}")

## 3 · Does each file carry values?

For each file, read a few columns at a time (the grid files have about 30 million rows):

1. every row names its place, and every month has the same number of places;
2. every forecast month has rows with usable draws, and every recorded month has recorded deaths;
3. the forecast rows' summaries are complete and in order: chances between 0 and 1 that fall as the threshold rises, expected deaths not negative, every row from the same number of draws (and the number you expect, if you set `CRAFD_EXPECTED_DRAWS`).

Then, across the nine files: the same forecast month, the same months, and the same number of places per level.

Rows with no usable draws are allowed: their summaries are empty by design (`n_draws` 0). A whole month of them is not.

In [ ]:
results, summaries = [], {}
for level in LEVELS:
    for outcome in OUTCOMES:
        if (level, outcome) not in paths or not paths[level, outcome].exists():
            results.append((level, outcome, "FAIL", "not downloaded: run the download cell above again"))
for (level, outcome), path in paths.items():
    if not path.exists():
        continue
    fc_month = made_in[level, outcome]
    place = PLACE_ID[level]
    try:
        names = pq.ParquetFile(path).schema_arrow.names
    except (OSError, ValueError) as unreadable:   # not a Parquet file, or cut short
        results.append((level, outcome, "FAIL", f"the file cannot be read ({type(unreadable).__name__}); download it again"))
        continue
    try:
        # Read only the columns the file has: a missing one is reported by the check, by name.
        wanted = ["month", place, "n_draws", f"{outcome}_actual"]
        table = pd.read_parquet(path, columns=[c for c in wanted if c in names])
        # Every file is compared with the others below, even one that fails its own checks.
        summaries[level, outcome] = {
            "made_in": fc_month,
            "months": sorted(table["month"].astype(str).unique()) if "month" in table else [],
            "places": int(table[place].nunique()) if place in table else 0,
        }
        results.append((level, outcome, "PASS", check_bulk_file_complete(table, level, outcome, fc_month)))
        del table
        summary_columns = [c for c in names if c.startswith(f"{outcome}_p_gt")] + [
            c for c in (f"{outcome}_mean", "n_draws", "month") if c in names]
        forecast_rows = pd.read_parquet(path, columns=summary_columns, filters=[("month", ">=", fc_month)])
        results.append((level, outcome, "PASS", check_forecast_summaries(forecast_rows, outcome, fc_month, EXPECTED_DRAWS)))
        del forecast_rows
    except DataCheckError as problem:
        results.append((level, outcome, "FAIL", str(problem)))
    except (OSError, ValueError) as unreadable:   # the file's contents cannot be read past its header
        results.append((level, outcome, "FAIL", f"the file cannot be read ({type(unreadable).__name__}); download it again"))
try:
    results.append(("all", "", "PASS", check_files_agree(summaries)))
except DataCheckError as problem:
    results.append(("all", "", "FAIL", str(problem)))

for level, outcome, verdict, detail in results:
    print(f"{verdict}  {level:8} {outcome:3} {detail}")
failed = [r for r in results if r[2] == "FAIL"]
if failed:
    raise SystemExit(f"The download is not complete: {len(failed)} check(s) failed. Tell VIEWS, with the lines marked FAIL.")
print(f"\nThe download is complete: {len(LEVELS) * len(OUTCOMES)} files in {DATA_DIR}, all checks passed.")

# Part B · What you are holding

Read once.

**Nine files, one table each.** A file holds every place at its level and every month: the recorded months first (with the recorded deaths in `{outcome}_actual` and empty summaries), then the forecast months (with the summaries and an empty `{outcome}_actual`). The forecast starts in the month the `X-Forecast-Made-In` header names; every row before it is a recorded month.

**The three levels.** `grid` is PRIO-GRID cells (about 55 × 55 km at the equator), with `latitude` and `longitude`; `admin1` is GAUL admin-1 regions; `country` is GAUL countries, with ISO3 codes beside them. Each grid row also says which region and country it is in.

**The summaries, per outcome** (`sb` state-based, `ns` non-state, `os` one-sided): the chances of more than 0, 25, 100, 500 and 1,000 deaths, the expected number of deaths, the deaths if there are any, a severe case, and `n_draws`, how many draws they come from. Every column is explained in the [data dictionary](../docs/api/data_dictionary.md).

**Empty means no answer, not zero.** A forecast row with no usable draws has empty summaries; the last recorded month is only partly recorded.

**Adding up yourself.** Expected deaths add up across places; chances do not. The data dictionary explains why, under "Adding up places or months yourself".

In [ ]:
# What each level's files look like: rows, places, months, and the columns of the first outcome.
for level in LEVELS:
    s = summaries.get((level, OUTCOMES[0]))
    if not s or not s["months"]:
        continue
    names = pq.ParquetFile(paths[level, OUTCOMES[0]]).schema_arrow.names
    print(f"{level:8} {s['places']:>7,} places x {len(s['months'])} months "
          f"({s['months'][0]} to {s['months'][-1]}), {len(names)} columns: {', '.join(names)}")